# Keyword-Based Disease Annotator

Annotate `data/processed/merged_data.csv` using keyword presence in `cleaned_text`.

**Label order contract:** `["AURI", "PN", "TB", "COVID"]`

This notebook:
- loads base keyword CSVs from `docs/keywords/`
- normalizes keywords and `cleaned_text`
- creates `cleaned_text`, `disease`, `misinformation`, and `sentiment` columns
- keeps no-disease rows as `[0,0,0,0]`
- writes `data/training_data/merged_data_annotated.csv`


In [1]:
import csv
import json
from pathlib import Path

import pandas as pd

LABELS = ["AURI", "PN", "TB", "COVID"]
OUTPUT_COLUMNS = ["cleaned_text", "disease", "misinformation", "sentiment"]
CWD = Path.cwd()
ROOT_DIR = next((path for path in [CWD, *CWD.parents] if (path / ".git").exists()), CWD)
KEYWORD_FILES = {
    "AURI": ROOT_DIR / "docs/keywords/ri_keywords.csv",
    "PN": ROOT_DIR / "docs/keywords/pn_keywords.csv",
    "TB": ROOT_DIR / "docs/keywords/tb_keywords.csv",
    "COVID": ROOT_DIR / "docs/keywords/covid_keywords.csv",
}
INPUT_PATH = ROOT_DIR / "data/processed/merged_data.csv"
OUTPUT_PATH = ROOT_DIR / "data/training_data/merged_data_annotated.csv"


In [2]:
def normalize_text(value):
    if value is None:
        return ""
    text = str(value).strip().lower()
    return "" if text == "nan" else text


def load_keywords(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing keyword file: {path}")

    keywords = []
    seen = set()

    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        for row in reader:
            for cell in row:
                keyword = normalize_text(cell)
                if keyword and keyword not in seen:
                    keywords.append(keyword)
                    seen.add(keyword)

    if not keywords:
        raise ValueError(f"Keyword file is empty after normalization: {path}")

    return keywords


def annotate_text(cleaned_text, keyword_map):
    text = normalize_text(cleaned_text)
    vector = [int(any(keyword in text for keyword in keyword_map[label])) for label in LABELS]
    return vector


In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {INPUT_PATH}")

with INPUT_PATH.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    fieldnames = reader.fieldnames or []

if "cleaned_text" not in fieldnames:
    raise KeyError("Expected `cleaned_text` column in the source dataset.")

keyword_map = {label: load_keywords(path) for label, path in KEYWORD_FILES.items()}
keyword_counts = {label: len(keywords) for label, keywords in keyword_map.items()}

print({
    "cwd": str(CWD),
    "root_dir": str(ROOT_DIR),
    "rows": len(rows),
    "input_path": str(INPUT_PATH),
    "output_path": str(OUTPUT_PATH),
    "keyword_counts": keyword_counts,
})


{'cwd': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/notebooks', 'root_dir': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus', 'rows': 30357, 'input_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/processed/merged_data.csv', 'output_path': '/Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/training_data/merged_data_annotated.csv', 'keyword_counts': {'AURI': 71, 'PN': 80, 'TB': 78, 'COVID': 115}}


In [4]:
annotated_rows = []
for row in rows:
    vector = annotate_text(row.get("cleaned_text", ""), keyword_map)
    annotated_rows.append({
        "cleaned_text": row.get("cleaned_text", ""),
        "disease": json.dumps(vector, separators=(",", ":")),
        "misinformation": 0,
        "sentiment": 0,
    })

annotated_df = pd.DataFrame(annotated_rows, columns=OUTPUT_COLUMNS)

print({"output_rows": len(annotated_df), "output_columns": list(annotated_df.columns)})
display(annotated_df.head())


{'output_rows': 30357, 'output_columns': ['cleaned_text', 'disease', 'misinformation', 'sentiment']}


,cleaned_text,disease,misinformation,sentiment
0,anong gusto mo gawin niya makipagbarda sa mga ...,"[1,1,1,1]",0,0
1,thoughts on the officiating crew tonight? that...,"[0,0,0,0]",0,0
2,horrible game when it comes to the vibe and a ...,"[0,0,0,0]",0,0
3,shit game. theres no other way to put it. refs...,"[0,0,0,0]",0,0
4,nixs accuracy on deep throws was ass.,"[0,0,0,0]",0,0


In [5]:
parsed_vectors = annotated_df["disease"].apply(json.loads).tolist()
vector_lengths = [len(values) for values in parsed_vectors]
binary_ok = [all(item in (0, 1) for item in values) for values in parsed_vectors]

assert list(annotated_df.columns) == OUTPUT_COLUMNS
assert len(annotated_df) == len(rows)
assert all(length == len(LABELS) for length in vector_lengths)
assert all(binary_ok)
assert annotated_df["misinformation"].isin([0, 1]).all()
assert annotated_df["sentiment"].isin([0, 1]).all()

print("Annotation contract validated.")
print({"input_rows": len(rows), "output_rows": len(annotated_df)})
print(pd.DataFrame(parsed_vectors, columns=LABELS).sum().astype(int).to_dict())


Annotation contract validated.
{'input_rows': 30357, 'output_rows': 30357}
{'AURI': 20475, 'PN': 6775, 'TB': 8296, 'COVID': 10243}


In [6]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
annotated_df.to_csv(OUTPUT_PATH, index=False)
print(f"Exported {len(annotated_df):,} annotated rows to {OUTPUT_PATH}")


Exported 30,357 annotated rows to /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/training_data/merged_data_annotated.csv


In [7]:
# Spot checks for single-label rows by disease.
eda_columns = ["cleaned_text", "disease", "misinformation", "sentiment"]
disease_vectors = pd.DataFrame(annotated_df["disease"].apply(json.loads).tolist(), columns=LABELS)
single_label_mask = disease_vectors.sum(axis=1).eq(1)

def sample_label_only(label, n=3):
    label_mask = disease_vectors[label].eq(1) & single_label_mask
    label_rows = annotated_df.loc[label_mask, eda_columns]
    return label_rows.sample(n=min(n, len(label_rows)), random_state=42)

label_sample_dfs = {label: sample_label_only(label) for label in LABELS}

for label, label_sample_df in label_sample_dfs.items():
    print(f"{label}-only examples")
    display(label_sample_df)


AURI-only examples


,cleaned_text,disease,misinformation,sentiment
5987,hi react ka sendan kita vids namin ng kapatid ko,"[1,0,0,0]",0,0
8860,sense of self bukod sa pagiging nanay hello mo...,"[1,0,0,0]",0,0
23175,sharing my experience as a former nursing stud...,"[1,0,0,0]",0,0


PN-only examples


,cleaned_text,disease,misinformation,sentiment
2718,feeling tired,"[0,1,0,0]",0,0
2826,rt : summer really got me sweating like im doi...,"[0,1,0,0]",0,0
20389,cafe recommendations please hi! college studen...,"[0,1,0,0]",0,0


TB-only examples


,cleaned_text,disease,misinformation,sentiment
16149,episode 3 : lesser known low cost remedies fro...,"[0,0,1,0]",0,0
5553,emily bront did not write her one and only (go...,"[0,0,1,0]",0,0
3802,house and lot bungalow for sale 192 sqm land a...,"[0,0,1,0]",0,0


COVID-only examples


,cleaned_text,disease,misinformation,sentiment
22364,diarrhea in my kitten [3 months old] hi. just ...,"[0,0,0,1]",0,0
7856,milan reports 50% of passengers in flights fro...,"[0,0,0,1]",0,0
6377,themillienb 03/05/26 so my kids live with thei...,"[0,0,0,1]",0,0
